# FluxPlayer - build the APK on Colab

Open https://colab.research.google.com, choose File -> Upload notebook, pick this file, then
Runtime -> Run all. A CPU runtime is enough.

It installs JDK 17 and the Android SDK (API 37 canary + API 34, build-tools 36 and 34), clones
1234567890933/fluxplayer, runs the unit tests, builds :app:assembleDebug and gives you the APK.

Colab has unrestricted internet, so jitpack and the Gradle plugin portal both work here. Those
are the two hosts that block the local sandbox, which is why this build succeeds where that one
could not.


In [ ]:
#@title Build FluxPlayer APK (run me)
# Colab: Runtime -> Change runtime type -> CPU is enough (2 vCPU / ~12 GB RAM).
# Takes about 12-18 minutes: SDK download, Gradle download, then the build.

import os, subprocess, pathlib

def sh(cmd, check=True):
    print("\n$ " + (cmd[:200] + ("..." if len(cmd) > 200 else "")))
    r = subprocess.run(cmd, shell=True, executable="/bin/bash", text=True, capture_output=True)
    out = (r.stdout or "") + (r.stderr or "")
    print(out[-4000:])
    if check and r.returncode != 0:
        raise SystemExit("FAILED (%d): %s" % (r.returncode, cmd[:120]))
    return out

# 1. JDK 17
sh("apt-get -qq update && apt-get -qq install -y openjdk-17-jdk-headless unzip curl > /dev/null")
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]
sh("java -version")

# 2. Android SDK. API 37 is not in the stable channel yet: the canary platform reports
#    ApiLevel 37.1 and is what AGP resolves for compileSdk 37.0.
sdk = "/content/android-sdk"
os.environ["ANDROID_HOME"] = os.environ["ANDROID_SDK_ROOT"] = sdk
os.makedirs(sdk + "/licenses", exist_ok=True)
pathlib.Path(sdk + "/licenses/android-sdk-license").write_text(
    "\n24333f8a63b6825ea9c5514f83c2829b004d1fee\nd56f5187479451eabf01fb78af6dfcb131a6481e\n")
pathlib.Path(sdk + "/licenses/android-sdk-preview-license").write_text(
    "\n8933bad161af4178b1185d1a37fbf41ea5269c55\n")
sh("mkdir -p %s/platforms/android-37.0 %s/platforms/android-34 %s/build-tools" % (sdk, sdk, sdk))
sh("cd /tmp && curl -sL -o p37.zip https://dl.google.com/android/repository/platform-CANARY_r15.zip "
   "&& unzip -q -o p37.zip -d p37 && cp -r p37/android-*/* %s/platforms/android-37.0/" % sdk)
sh("cd /tmp && curl -sL -o p34.zip https://dl.google.com/android/repository/platform-34-ext7_r03.zip "
   "&& unzip -q -o p34.zip -d p34 && cp -r p34/android-*/* %s/platforms/android-34/" % sdk)
for ver, zf in [("36.0.0", "build-tools_r36_linux.zip"), ("34.0.0", "build-tools_r34-linux.zip")]:
    sh("cd /tmp && curl -sL -o bt.zip https://dl.google.com/android/repository/%s "
       "&& unzip -q -o bt.zip -d btx && mkdir -p %s/build-tools/%s && cp -r btx/android-*/* %s/build-tools/%s/"
       % (zf, sdk, ver, sdk, ver))
sh("ls %s/platforms %s/build-tools" % (sdk, sdk))

# 3. Repository
sh("cd /content && rm -rf fluxplayer && git clone --depth 1 https://github.com/1234567890933/fluxplayer.git")
sh("cd /content/fluxplayer && echo sdk.dir=%s > local.properties && chmod +x gradlew && cat local.properties" % sdk)

# 4. Unit tests. Deliberately non-fatal: a failing test must not stop the APK from being
#    produced, so the failures are collected and printed instead of aborting here.
print("\n=== unit tests (non-fatal) ===")
test_out = sh("cd /content/fluxplayer && ./gradlew test --console=plain --no-daemon", check=False)
fails = [l for l in test_out.splitlines() if "FAILED" in l]
print("\n--- test failures: %d ---" % len(fails))
for l in fails[:25]:
    print(l)

# 5. Build
print("\n=== assembleDebug ===")
sh("cd /content/fluxplayer && ./gradlew :app:assembleDebug --console=plain --no-daemon")

# 6. Collect the APKs
sh("cd /content/fluxplayer && mkdir -p /content/out && "
   "find . -name '*.apk' -path '*outputs/apk*' -exec cp {} /content/out/ \\; && ls -lh /content/out")
print("\nDONE - run the next cell to download the APK.")


In [ ]:
#@title Download the APK
import glob, os
from google.colab import files

apks = sorted(glob.glob("/content/out/*.apk"))
if not apks:
    print("No APK found - read the build output above for the first error line.")
else:
    for a in apks:
        print("%s  %.1f MB" % (a, os.path.getsize(a) / 1e6))
        files.download(a)
